In [ ]:
from xmlrpc.server import SimpleXMLRPCServer, SimpleXMLRPCRequestHandler
import datetime
import sqlite3
import threading

class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

server = SimpleXMLRPCServer(("localhost", 8018),
                            requestHandler=RequestHandler,
                            allow_none=True)
server.register_introspection_functions()

DB_FILE = 'log.db'
lock = threading.Lock()

def init_db():
    with lock:
        conn = sqlite3.connect(DB_FILE)
        cur = conn.cursor()
        cur.execute('''
            CREATE TABLE IF NOT EXISTS events (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                event_type TEXT NOT NULL,
                timestamp TEXT NOT NULL,
                duration REAL
            )
        ''')
        conn.commit()
        conn.close()

init_db()

def ping_stats():
    return "stats2_ok"

def add_log(event_type, duration=None):
    with lock:
        conn = sqlite3.connect(DB_FILE)
        cur = conn.cursor()
        ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        cur.execute(
            "INSERT INTO events (event_type, timestamp, duration) VALUES (?, ?, ?)",
            (event_type, ts, duration)
        )
        conn.commit()
        conn.close()
    return True

def get_log(event_type=None, dt_from=None, dt_to=None, min_duration=None, max_duration=None):
    conn = sqlite3.connect(DB_FILE)
    cur = conn.cursor()
    q = "SELECT event_type, timestamp, duration FROM events WHERE 1=1"
    p = []
    if event_type:
        q += " AND event_type = ?"; p.append(event_type)
    if dt_from:
        q += " AND timestamp >= ?"; p.append(dt_from)
    if dt_to:
        q += " AND timestamp <= ?"; p.append(dt_to)
    if min_duration is not None:
        q += " AND duration >= ?"; p.append(min_duration)
    if max_duration is not None:
        q += " AND duration <= ?"; p.append(max_duration)
    q += " ORDER BY timestamp ASC"
    cur.execute(q, p)
    rows = cur.fetchall()
    conn.close()
    return rows

server.register_function(ping_stats, 'ping_stats')
server.register_function(add_log, 'add_log')
server.register_function(get_log, 'get_log')

print("Stats2 server on port 8018 (SQLite)...")
server.serve_forever()

Stats2 server on port 8018 (SQLite)...


127.0.0.1 - - [19/Oct/2026 18:30:19] "POST /RPC2 HTTP/1.1" 200 -
